In [1]:
from src.improved_model import SimpleCNN
import torch.optim as optim
import matplotlib.pyplot as plt

from src.load_and_save import save_model
from src.model import SimpleNN
from src.pruning import get_intermediate_outputs_as_numpy
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

from src.pruning import is_all_layers_separated
import numpy as np


from src.load_and_save import load_model

cnn_model = SimpleCNN()
cnn_model.load_state_dict(torch.load(settings.models_path / 'convnet.pth'))
cnn_model.to(device)


C:\Users\frrit\AppData\Local\Temp\ipykernel_33924\1988482374.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(settings.models_path /

SimpleCNN(
  (layer1): Conv2d(1, 8, kernel_size=(5, 5), stride=(5, 5), padding=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer2): Linear(in_features=288, out_features=54, bias=True)
  (layer3): Linear(in_features=54, out_features=10, bias=True)
)

In [2]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=5000, shuffle=False)

In [3]:
test_data, _  = next(iter(test_dataloader))
test_data = test_data.to(device)
cnn_model.eval_mode()


In [4]:
input_binary = cnn_model.first_layer(test_data).detach().cpu().numpy().astype(np.int8)
output_binary = cnn_model.second_layer(cnn_model.first_layer(test_data)).detach().cpu().numpy().astype(np.int8)
# input_binary.s
print(input_binary.shape, output_binary.shape)
print(cnn_model.layer2.weight.shape)

weights = cnn_model.layer2.weight.T.detach().cpu().numpy()
bias = cnn_model.layer2.bias.T.detach().cpu().numpy()


(5000, 288) (5000, 54)
torch.Size([54, 288])


C:\Users\frrit\AppData\Local\Temp\ipykernel_33924\3880587837.py:8: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3701.)
  bias = cnn_model.layer2.bias.T.detach().cpu().numpy()


In [16]:
from collections import Counter


select_row: int = 0
X = input_binary
Y = output_binary[:, select_row]
W = weights[:, select_row]

# Remove columns where .99 fraction of values are identical:
threshold = .999
def most_common_fraction(column):
    counts = Counter(column)
    most_common_count = counts.most_common(1)[0][1]  # Frequency of the most common value
    return most_common_count / len(column)

# Apply to each column
fractions = np.apply_along_axis(most_common_fraction, axis=0, arr=X)

# Identify columns to keep
columns_to_keep = fractions < threshold
print(f"Columns rejected: {X.shape[1] - sum(columns_to_keep)}")

# Filter the columns
X = X[:, columns_to_keep]
W = W[columns_to_keep]

# Keep only unique rows:
X, indices = np.unique(X, axis=0, return_index=True)
Y = Y[indices]

y_rec = X @ W
assert np.min(y_rec[Y==1]) > np.max(y_rec[Y==-1])
B = - (np.min(y_rec[Y==1]) + np.max(y_rec[Y==-1])) / 2

assert all(np.sign(X @ W + B) == Y)
# X @ W + B == Y

Columns rejected: 65


In [6]:
import pulp
import numpy as np

# Example Input: Replace these with your actual inputs
X = X[:300,:]#np.random.choice([1, -1], size=(1000, 50))  # Large matrix, 1000 samples x 50 features
Y = Y[:300]#np.random.choice([1, -1], size=(1000,))    # Labels, 1000 samples

# Problem dimensions
n, m = X.shape

# Initialize the ILP problem
model = pulp.LpProblem("Threshold_Classification_Relaxed", pulp.LpMinimize)

# Define binary variables A_plus and A_minus for each feature
A_plus = [pulp.LpVariable(f"A_plus_{j}", cat="Binary") for j in range(m)]
A_minus = [pulp.LpVariable(f"A_minus_{j}", cat="Binary") for j in range(m)]

# Define the actual vector A in terms of A_plus and A_minus
A = [A_plus[j] - A_minus[j] for j in range(m)]

# Integer variable for threshold k
k = pulp.LpVariable("k", cat="Integer")

# Binary slack variables z to allow misclassifications
z = [pulp.LpVariable(f"z_{i}", cat="Binary") for i in range(n)]

# Objective: Minimize the number of misclassifications
model += pulp.lpSum(z), "Minimize_misclassifications"

# Add constraints for each data point
M = 1000  # A large constant for the big-M method
for i in range(n):
    # Compute s_i = X[i] * A
    s_i = pulp.lpSum(X[i, j] * A[j] for j in range(m))

    if Y[i] == -1:
        # Constraint for Y[i] = -1: X[i] * A <= k
        model += s_i <= k + M * z[i], f"Constraint_neg_{i}"
    else:
        # Constraint for Y[i] = 1: X[i] * A > k
        model += s_i >= k + 1 - M * z[i], f"Constraint_pos_{i}"

# Solve the problem using the CBC solver
solver = pulp.PULP_CBC_CMD(msg=True)
status = model.solve(solver)

# Extract and display results
if pulp.LpStatus[status] == "Optimal":
    A_solution = [pulp.value(A[j]) for j in range(m)]
    k_solution = pulp.value(k)
    misclassifications = sum(pulp.value(z[i]) for i in range(n))

    print("Optimal Solution Found!")
    print("A:", A_solution)
    print("k:", k_solution)
    print("Misclassifications:", misclassifications)

    # Final Step: Convert A and k to NumPy arrays
    A_numpy = np.array(A_solution)
    k_numpy = np.array(k_solution)

    print("A_numpy:", A_numpy)
    print("k_numpy:", k_numpy)

    # Post-process: Round A to {-1, 0, 1}
    A_rounded = np.sign(A_numpy)  # Round A to {-1, 0, 1}

    # Compute the prediction
    prediction = np.sign(X @ A_rounded + k_numpy)  # Compute the predicted labels

    # Check the accuracy
    accuracy = np.sum(prediction == Y) / n
    print("Accuracy:", accuracy)
else:
    print("No feasible solution found.")


Optimal Solution Found!
A: [0.0, 0.0, 0.0, 0.0, 0.0, -1.0, 0.0, -1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, -1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, -1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, -1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, -1.0, 0.0, -1.0, -1.0, 0.0, 1.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, -1.0, -1.0, -1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0, 0.0, -1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, -1.0, 0.0, 0.0, 0.0, 0.0, -1.0, 1.0, -1.0, -1.0, -1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, -1.0, -1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, -1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, -1.0, 0.0, -1.0, 0.0, 0.0, 0.0, 0.0, -1.0, -1.0, 0.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0, -1.0, -1.0, 0.0, 0.0, 0.0, -1.0, -1.0, 0.0, 0.0, 

0.5811948676824379

In [71]:
from src.model import scramble_activation
import torch

# Inputs: X (n x m matrix), Y (vector of size n)
X = torch.Tensor(X)
Y = torch.Tensor(Y)

# Model parameters: Initialize A (m-dim vector) and k (scalar)
m = X.shape[1]
A = torch.randn(m, requires_grad=True)  # Random initialization

# Hyperparameters
learning_rate = 0.01
num_epochs = int(1e4)
batch_size = 32  # Set appropriate batch size

# Optimizer
optimizer = torch.optim.Adam([A], lr=learning_rate)

# Training loop with batching
for epoch in range(num_epochs):
    # Shuffle the data (for batching)
    perm = torch.randperm(X.size(0))  # Random shuffle indices
    X_shuffled = X[perm]
    Y_shuffled = Y[perm]

    # Process data in batches
    for i in range(0, X.size(0), batch_size):
        # Get the current batch
        X_batch = X_shuffled[i:i + batch_size]
        Y_batch = Y_shuffled[i:i + batch_size]

        # Compute scores: X * A
        scores = X_batch @ scramble_activation(A, True, 3.0)  # Shape: (batch_size,)

        # Compute losses
        loss_neg = torch.relu(scores) * (Y_batch == -1)  # Penalize if X*A > k for Y=-1
        loss_pos = torch.relu(-scores) * (Y_batch == 1)  # Penalize if X*A <= k for Y=1

        # Total loss: Sum of both losses
        total_loss = torch.sum(loss_neg + loss_pos)

        # Backward pass and optimization step
        optimizer.zero_grad()  # Zero gradients
        total_loss.backward()  # Compute gradients
        optimizer.step()  # Update parameters

    # Print progress every 100 epochs
    if epoch % int(1e2) == 0:
        print(f"Epoch {epoch}: Loss = {total_loss.item()}")

# Post-processing: Threshold A to {-1, 1}
A_final = torch.sign(A).detach()  # Convert to binary vector
k_final = k.detach()

# Evaluate on the entire dataset
scores = X @ A_final  # Shape: (n,)
print(total_loss)

# Results
print("Optimized A:", A_final)


Epoch 0: Loss = 74.25248718261719
Epoch 100: Loss = 2.707606315612793
Epoch 200: Loss = 32.32268524169922
Epoch 300: Loss = 13.483308792114258
Epoch 400: Loss = 21.026226043701172
Epoch 500: Loss = 15.427964210510254
Epoch 600: Loss = 11.360504150390625
Epoch 700: Loss = 6.880797863006592
Epoch 800: Loss = 5.629494667053223
Epoch 900: Loss = 27.003395080566406
Epoch 1000: Loss = 11.495176315307617
Epoch 1100: Loss = 11.398712158203125
Epoch 1200: Loss = 9.007696151733398
Epoch 1300: Loss = 0.0
Epoch 1400: Loss = 2.6332082748413086
Epoch 1500: Loss = 7.17667818069458
Epoch 1600: Loss = 1.1428711414337158
Epoch 1700: Loss = 11.948799133300781
Epoch 1800: Loss = 2.488595962524414
Epoch 1900: Loss = 9.007506370544434
Epoch 2000: Loss = 14.381830215454102
Epoch 2100: Loss = 3.832566261291504
Epoch 2200: Loss = 5.797895431518555
Epoch 2300: Loss = 13.256088256835938
Epoch 2400: Loss = 3.493840217590332
Epoch 2500: Loss = 7.230388164520264
Epoch 2600: Loss = 5.028763771057129
Epoch 2700: Loss

KeyboardInterrupt: 

In [72]:
# Post-processing: Threshold A to {-1, 1}
A_final = torch.sign(A).detach()  # Convert to binary vector

torch.sum(torch.sign(X @ A_final) == Y)/len(Y)
# k_final

tensor(0.8759)

In [97]:
def get_y_reconstructed(a: np.ndarray, x: np.ndarray) -> np.ndarray:
    return x @ np.sign(a)

def get_threshold(y_rec, y) -> int:
    y_rec_positive = y_rec[y == 1]
    return min(y_rec_positive)


# Get score:
def get_score(y: np.ndarray, y_rec: np.ndarray, threshold: int) -> float:
    Y_score = np.sign(y_rec - (threshold - .5))
    return sum(Y_score == y) / len(y)


# y_target = Y.copy()
# A = pinv_X @ y_target

A = np.random.randint(0,2,X.shape[1])*2 - 1

y_rec = get_y_reconstructed(A, X)
threshold = get_threshold(y_rec, Y)
score = get_score(Y, y_rec, threshold)
print(score)

def get_y_target(y_rec, Y, threshold) -> np.ndarray:
    y_target = y_rec.copy()
    y_rec_positive = y_rec[Y == 1]
    y_rec_negative = y_rec[Y == -1]
    y_rec_positive = np.maximum(y_rec_positive, threshold)
    y_rec_negative = np.minimum(y_rec_negative, threshold - 1)
    y_target[Y==1] = y_rec_positive
    y_target[Y==-1] = y_rec_negative
    return y_target
#y_target = Y.copy()
for i in range(5):
    y_target = get_y_target(y_rec, Y, threshold)
    A = pinv_X @ y_target
    y_rec = get_y_reconstructed(A, X)
    threshold = get_threshold(y_rec, Y)
    score = get_score(Y, y_rec, threshold)
    print(score)



0.521484375
0.53662109375
0.53564453125
0.53564453125
0.53564453125
0.53564453125


In [175]:
y_positive = Y[Y==1] * 0
x_positive = X[Y==1, :]
y_negative = Y[Y==-1] * 0
x_negative = X[Y==-1, :]


A = np.zeros(shape=[X.shape[1]])

record = 0.0

x_positive_sub_set = x_positive[y_positive == min(y_positive),:]
x_negative_sub_set = x_negative[y_negative == max(y_negative),:]
for index in np.where(A == 0)[0]:
    if abs(np.mean(x_positive_sub_set[:,index]) - np.mean(x_negative_sub_set[:,index])) > record:
        record = abs(np.mean(x_positive_sub_set[:,index]) - np.mean(x_negative_sub_set[:,index]))
        A[index] = 1
    else:
        A[index] = -1
# for index in range(X.shape[1]):
#     if all(x_positive[:,index] == x_positive[0,index]):
#         if not all(x_negative[:,index] == x_positive[0,index]):
#             print(index, "Unique for negative")
#             print(np.mean(x_negative[:,index]))
#
#     if all(x_negative[:,index] == x_negative[0,index]):
#         if not all(x_positive[:,index] == x_negative[0,index]):
#             print(index, "Unique for positive")
#             print(np.mean(x_positive[:,index]))

0.03339732942444096 0
0.19508288470592172 1
0.27329958993984027 2
0.2989130539789103 7
0.3369191324268258 8
0.46855185712105174 9
0.7343915337100108 25


In [173]:
np.mean(x_negative[:,25])
np.mean(x_positive[:,25])


-0.513713862120089